# GPT-2 — Language Models are Unsupervised Multitask Learners (2019)

_Papers · Transformers & LLMs_

**Stack causal (masked) Transformer blocks, train one objective — predict the next token — at scale, and the model learns many tasks with no task-specific training.**

---

This notebook is a practice scaffold for the **GPT-2 — Language Models are Unsupervised Multitask Learners (2019)** lesson. We build a tiny nanoGPT one piece at a time: first the two worked examples (the causal mask and next-token cross-entropy), then masked self-attention, then a pre-norm block and the full model, and finally a char-level training loop. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Two worked examples: the causal mask and next-token loss

Two ideas drive GPT. **(a) The causal mask**: when computing attention, each token may look only at itself and earlier tokens — never the future. We set scores above the diagonal to `−inf` before the softmax, so token 0 attends only to itself and token 1 splits its attention over tokens 0 and 1.

**(b) Next-token cross-entropy**: the training loss is `−log p(true next token)`. We print the softmax probabilities and the loss for the correct token, and for a wrong token, to see how the loss grows when the model is confidently wrong.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

# (a) Causal mask: 3 tokens, 1 head, d_k=2, identity Q=K=V=X
X = torch.tensor([[1., 0.],
                  [0., 1.],
                  [1., 1.]])                      # 3 tokens, d_k=2
S = X.shape[0]
scores = X @ X.transpose(-2, -1) / math.sqrt(2)   # raw scaled scores
mask = torch.triu(torch.ones(S, S), diagonal=1).bool()   # True ABOVE the diagonal = the future
scores = scores.masked_fill(mask, float("-inf"))  # forbid attending to j > t (BEFORE softmax)
attn = F.softmax(scores, dim=-1)                  # token 0 -> [1,0,0]; token 1 -> [.33,.67,0]
out = attn @ X
print("token0 weights =", [round(v, 4) for v in attn[0].tolist()])   # [1.0, 0.0, 0.0]
print("token1 weights =", [round(v, 4) for v in attn[1].tolist()])   # [0.3302, 0.6698, 0.0]
print("token2 output  =", [round(v, 4) for v in out[2].tolist()])    # [0.7517, 0.7517]

# (b) Next-token cross-entropy: vocab {0,1,2,3}, true next token = 2
z = torch.tensor([1.0, 0.0, 3.0, 0.5])
p = F.softmax(z, dim=-1)
print("softmax probs  =", [round(v, 4) for v in p.tolist()])         # [0.1068, 0.0393, 0.7891, 0.0648]
loss_true2 = -torch.log(p[2])
print("CE loss (true=2) =", round(loss_true2.item(), 4))             # 0.2368
print("CE loss (true=0) =", round((-torch.log(p[0])).item(), 4))     # 2.2363  (practice problem)

### Step 2 — Masked multi-head self-attention

This is the transformer's attention with the causal mask baked in. We project the input to queries, keys and values, split them across heads, and compute scaled dot-product scores.

Before the softmax we apply the precomputed upper-triangular mask so no position can attend to the future. The per-head outputs are concatenated and passed through an output projection.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, h, max_len):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # precompute the causal mask once (True above diagonal = the future)
        self.register_buffer("mask", torch.triu(torch.ones(max_len, max_len), diagonal=1).bool())

    def _split(self, x):                          # (B,S,d) -> (B,h,S,d_k)
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self._split(self.W_q(x))
        K = self._split(self.W_k(x))
        V = self._split(self.W_v(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)        # scaled dot product
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf")) # THE causal mask: forbid the future
        out = F.softmax(scores, dim=-1) @ V                           # (B,h,S,d_k)
        out = out.transpose(1, 2).contiguous().view(B, S, self.h * self.d_k)  # concat heads
        return self.W_o(out)

### Step 3 — The pre-norm block and the full nanoGPT

GPT-2 uses **pre-norm** blocks: LayerNorm is applied at the *input* of each sub-block (§2.3), and the attention and feed-forward outputs are added back via residual connections.

The full model embeds tokens and adds **learned** positional embeddings (not sinusoids), runs N blocks, applies a final LayerNorm (GPT-2's extra one), and projects to vocab logits. `generate` samples tokens one at a time using the next-token distribution.

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, h, max_len, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, h, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # pre-norm residual around masked attention
        x = x + self.ff(self.ln2(x))       # pre-norm residual around feed-forward
        return x


class NanoGPT(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, n_layers=3, max_len=64, d_ff=128):
        super().__init__()
        self.max_len = max_len
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)        # LEARNED positions (GPT style, not sinusoids)
        self.blocks = nn.ModuleList([Block(d_model, h, max_len, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)                # GPT-2's extra final LayerNorm (S 2.3)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, idx):
        B, S = idx.shape
        pos = torch.arange(S, device=idx.device)
        x = self.tok(idx) + self.pos(pos)                # add learned positional embedding
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))                   # (B,S,vocab) logits

    @torch.no_grad()
    def generate(self, idx, n_new, temp=0.8):
        for _ in range(n_new):
            logits = self(idx[:, -self.max_len:])[:, -1, :] / temp   # next-token logits at the last position
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)            # SAMPLE (see mod-llm for temperature)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### Step 4 — Char-level corpus and next-token training

We build a tiny char-level vocabulary from a short repeated text. The training target is the input shifted left by one — at every position the model predicts the **next** character.

We train with cross-entropy (Eq. 1) for ~2000 steps, printing a sample every 500. The loss falls from about `ln(20) ≈ 3.0` (random over 20 chars) toward ~0.2, and the samples go from gibberish to readable Shakespeare-ish text. The trailing comment notes how removing the mask lets the model cheat at train time but breaks generation.

In [ ]:
# Tiny char-level corpus
text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB = len(chars)                                       # 20 here -> random-init loss ~ ln(20) ~ 3.0
data = torch.tensor([stoi[c] for c in text])
SEQ, B = 32, 64

def get_batch():
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ] for i in ix])
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])   # NEXT char at every position (shift by 1)
    return x, y

def sample(net):
    start = torch.tensor([[stoi["t"]]])
    out = net.generate(start, 60)[0].tolist()
    return "".join(itos[i] for i in out)

torch.manual_seed(0)
net = NanoGPT(VOCAB, max_len=SEQ)
opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
for step in range(2001):
    x, y = get_batch()
    logits = net(x)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))   # next-token cross-entropy (Eq. 1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}   sample: {sample(net)!r}")
# Loss falls from ~3.0 (random over 20 chars) toward ~0.2; samples go gibberish -> readable Shakespeare-ish.
# (Our small run, not the paper's reported number. Exact values vary by hardware/seed.)

# ABLATION: remove the causal mask -> loss drops LOWER (it cheats) but generation breaks.
# To run it, set self.mask to all-False in CausalSelfAttention (no future is forbidden), retrain, and
# call sample(): training loss looks great, but generated text is incoherent because at generation time
# there is no future token to copy. The CODEVIZ panel charts the masked run's falling loss + sample quality.

## Visualize the data & results

_As a tiny char-level nanoGPT trains with next-token cross-entropy (Eq. 1) and the causal mask, does the loss fall and does the generated text become readable — and does removing the mask break generation?_

### Step 1 — Rebuild a compact nanoGPT with a mask toggle

This cell is self-contained so the plot can run on its own. It re-defines a compact nanoGPT whose attention takes a `use_mask` flag, so we can train identical models with the causal mask ON and OFF and compare. All hyperparameters and seeds match the reference implementation.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)


class CSA(nn.Module):                       # causal self-attention (use_mask toggles the ablation)
    def __init__(self, d, h, mx, use_mask=True):
        super().__init__()
        self.h = h
        self.dk = d // h
        self.use_mask = use_mask
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
        self.register_buffer("m", torch.triu(torch.ones(mx, mx), 1).bool())

    def split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)

    def forward(self, x):
        B, S, _ = x.shape
        Q, K, V = self.split(self.Wq(x)), self.split(self.Wk(x)), self.split(self.Wv(x))
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if self.use_mask:
            sc = sc.masked_fill(self.m[:S, :S], float("-inf"))   # forbid the future
        a = F.softmax(sc, dim=-1) @ V
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))


class Block(nn.Module):
    def __init__(self, d, h, mx, ff, use_mask=True):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.a = CSA(d, h, mx, use_mask)
        self.n2 = nn.LayerNorm(d)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))

    def forward(self, x):
        x = x + self.a(self.n1(x))
        return x + self.f(self.n2(x))


class GPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=3, mx=32, ff=128, use_mask=True):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(mx, d)
        self.b = nn.ModuleList([Block(d, h, mx, ff, use_mask) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)

    def forward(self, idx):
        B, S = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(S))
        for blk in self.b:
            x = blk(x)
        return self.head(self.lnf(x))

### Step 2 — Train with the mask ON vs. OFF and compare curves

This cell rebuilds the same corpus and a `run` helper that trains for ~2000 steps, logging the loss every 100. We run it twice — mask ON and OFF — and print both curves.

The mask-ON loss falls to a useful minimum (~0.2) and would generate readable text. The mask-OFF loss falls *lower* (~0.02) because the model cheats by attending to the future token it is asked to predict — but at generation time there is no future to copy, so it is useless. That contrast is the whole point of the causal mask.

In [ ]:
text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
V = len(chars)
data = torch.tensor([stoi[c] for c in text])
SEQ, B = 32, 64
words = set(w for w in text.replace(",", " ").replace(".", " ").split() if w)

def batch():
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ] for i in ix])
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])
    return x, y

def run(use_mask, steps=2001):
    torch.manual_seed(0)
    net = GPT(V, use_mask=use_mask)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
    curve = []
    for s in range(steps):
        x, y = batch()
        lg = net(x)
        loss = F.cross_entropy(lg.reshape(-1, V), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 100 == 0:
            curve.append((s, round(loss.item(), 3)))
    return curve

print("mask ON :", run(True))
print("mask OFF:", run(False))   # falls lower (cheats), but generation would be incoherent
# mask ON -> loss ~3.0 -> ~0.2, samples become readable. mask OFF -> ~0.02 but useless to generate.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The causal-mask ablation. Your nanoGPT trains to low loss and generates readable text. You
            now remove the causal mask (let every position attend to the whole sequence, future
            included) and retrain. What happens to (i) the training loss and (ii) the quality of generated text,
            and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the masking line: stop setting future scores to $-\infty$; keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; the mask &mdash; so any difference is attributable to it._
- Watch the training loss: it drops faster and lower than the masked model. — _Without the mask, position $t$ can attend to position $t{+}1$ &mdash; which holds the answer. The model learns to copy the future token instead of predicting it. Training loss looks great because it is cheating._
- Now generate. Feed a prompt and sample one token at a time; the output is incoherent. — _At generation time the future does not exist yet, so the shortcut the model learned (read the next token) is unavailable. It never learned to predict from the past alone._

**Answer:** Removing the mask makes training loss drop faster and lower &mdash; but only because
                 the model cheats: position $t$ can attend to position $t{+}1$, which contains the very token it
                 is asked to predict, so it learns to copy rather than predict. At generation time there
                 is no future to copy, so the model produces gibberish. This shows the causal mask is not an
                 optimization detail &mdash; it is what makes the network a genuine language model
                 (Eq. 1's conditioning on the past only) rather than a copy machine. The CODEVIZ panel shows
                 the masked model's loss falling and its samples becoming readable.

</details>

**Problem 2.** In worked example (b), the head's logits were $z=[1.0, 0.0, 3.0, 0.5]$ over the vocab
            $\{0,1,2,3\}$ and the true next token was $2$, giving loss $\approx 0.237$. Suppose instead the
            true next token had been $0$. Recompute the loss, and explain what gradient signal that larger loss
            sends.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reuse the same softmax probabilities $p \approx [0.1068, 0.0393, 0.7891, 0.0648]$ (the logits did not change, only which token is "true"). — _Cross-entropy depends only on the probability the model assigned to the true token; the distribution is the same._
- The probability on token $0$ is $0.1068$, so the loss is $-\ln(0.1068) \approx 2.24$. — _The model put little mass on the correct token, so $-\ln p$ is large &mdash; about $9\times$ the loss when the answer was the confident token $2$._
- Note the gradient pushes the logit of token $0$ up and the others (especially $2$) down. — _Minimizing $-\ln p(\text{true})$ raises the true token's probability and lowers the rest; the bigger the loss, the bigger the push._

**Answer:** With true token $0$, the loss is $-\ln(0.1068) \approx \mathbf{2.24}$ &mdash; far larger
                 than the $0.237$ we got when the answer was the high-probability token $2$. The gradient of
                 $-\ln p(0)$ raises the logit for token $0$ and lowers the others (most of all the over-confident
                 token $2$), so after the step the model assigns more probability to the character that actually
                 came next. That is exactly how next-token cross-entropy teaches the language model.

</details>

**Problem 3.** GPT-2 (\S 2.3) scales residual-layer weights at initialization by $1/\sqrt{N}$, where $N$ is the
            number of residual layers. For the 1.5B model (Table 2: 48 layers) versus the 117M model (12
            layers), what are those factors, and why does a deeper stack want a smaller initial residual?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the factors: $1/\sqrt{48} \approx 0.144$ for the 48-layer model and $1/\sqrt{12} \approx 0.289$ for the 12-layer model. — _The paper ties the scale directly to the residual depth $N$, so deeper models start with smaller residual contributions._
- Recall that a residual stream accumulates: each block adds its output to a running sum, so after $N$ blocks the variance can grow with $N$. — _The paper says the init "accounts for the accumulation on the residual path" &mdash; more layers means more accumulation to control._
- Conclude that shrinking each block's initial contribution by $1/\sqrt{N}$ keeps the total residual signal at a sane scale at the start of training, regardless of depth. — _It stabilizes early training of very deep stacks, the same motivation as pre-norm._

**Answer:** The factors are $1/\sqrt{48} \approx 0.144$ (48-layer, 1.5B) and $1/\sqrt{12} \approx
                 0.289$ (12-layer, 117M) &mdash; the deeper model starts with smaller residual contributions.
                 Because a residual stream adds each block's output into a running sum, the accumulated signal
                 tends to grow with depth $N$; scaling the initial residual weights by $1/\sqrt{N}$ cancels
                 that growth so the network starts well-scaled no matter how deep it is. The paper introduces
                 this exactly "to account for the accumulation on the residual path" (\S 2.3). Our tiny model
                 is too shallow to need it, but it is why the big ones train stably.

</details>